### kroki 1 oraz 2

In [1]:
import pandas as pd
from google.cloud import bigquery
from google.oauth2 import service_account
import numpy as np

credentials = service_account.Credentials.from_service_account_file("key.json") # lokalizacja pobranego klucza

client = bigquery.Client(credentials=credentials, project=credentials.project_id) 

query = """
SELECT *
FROM `bigquery-public-data.noaa_gsod.gsod2020`
LIMIT 10
"""

query_job = client.query(query)
query_result = query_job.result()
df = query_result.to_dataframe()

### krok 3

In [2]:
sql_query = """
SELECT 
    obs.stn, obs.wban, obs.year, obs.mo, obs.da, obs.temp, obs.prcp, obs.max, obs.min, obs.visib,
    st.country
FROM (
    SELECT * FROM `bigquery-public-data.noaa_gsod.gsod1961`
    UNION ALL
    SELECT * FROM `bigquery-public-data.noaa_gsod.gsod1962`
) AS obs
LEFT JOIN `bigquery-public-data.noaa_gsod.stations` AS st
  ON obs.stn = st.usaf AND obs.wban = st.wban
"""

df = client.query(sql_query).to_dataframe()

print("3. Analiza struktury danych: ")
print(f"3.1. Liczba zapisanych wierszy: {len(df)}")
print(f"3.2. Liczba krajów: {df['country'].dropna().nunique()}")
print("\n3.3. Sposób zapisu dziennego dla lokalizacji:")
dzienny_podglad = (
    df.sort_values(['stn', 'wban', 'year', 'mo', 'da'])
      .groupby(['stn', 'wban'], as_index=False)
      .head(2)
      [['stn', 'wban', 'year', 'mo', 'da', 'temp', 'prcp']]
      .head(5)
)
print(dzienny_podglad)

print("\n3.4. Formaty zapisu wartości liczbowych:")
kolumny_numeryczne = df.select_dtypes(include='number').columns
print(df[kolumny_numeryczne].dtypes)

daty = pd.to_datetime(
    {
        'year': df['year'],
        'month': df['mo'],
        'day': df['da']
    }
)

temp_ok = df['temp'] < 900
prcp_ok = df['prcp'] < 90

print(f"\n3.5. Przedział czasowy: {daty.min().date()} - {daty.max().date()}")
print(f"Zakres dat dla temperatury: {daty[temp_ok].min().date()} - {daty[temp_ok].max().date()}")
print(f"Zakres dat dla opadów: {daty[prcp_ok].min().date()} - {daty[prcp_ok].max().date()}")

print("\n3.6. Dodatkowe informacje o danych pogodowych:")
print("1. Zbiór zawiera dane dobowe: rok (year), miesiąc (mo), dzień (da).")
print("2. Lokalizacja stacji jest identyfikowana przez parę: stn + wban.")
print("3. Kolumna country z tabeli stations opisuje kraj stacji.")
print("4. Temp, max i min opisują temperatury, a prcp oznacza opad dobowy.")
print("5. Kolumna visib przechowuje wartość widoczności.")
print("6. W danych NOAA braki często mają wartości zastępcze (np. 99.99, 999.9).")

3. Analiza struktury danych: 
3.1. Liczba zapisanych wierszy: 1863120
3.2. Liczba krajów: 144

3.3. Sposób zapisu dziennego dla lokalizacji:
            stn   wban  year  mo  da  temp  prcp
313450   028070  99999  1961  01  01  21.4  0.02
844487   028070  99999  1961  01  02  18.9  0.08
846322   028360  99999  1961  01  01  19.6  0.04
837529   028360  99999  1961  01  02  21.4  0.12
1594605  028640  99999  1961  01  01  22.3  0.04

3.4. Formaty zapisu wartości liczbowych:
temp     float64
prcp     float64
max      float64
min      float64
visib    float64
dtype: object

3.5. Przedział czasowy: 1961-01-01 - 1962-12-31
Zakres dat dla temperatury: 1961-01-01 - 1962-12-31
Zakres dat dla opadów: 1961-01-01 - 1962-12-31

3.6. Dodatkowe informacje o danych pogodowych:
1. Zbiór zawiera dane dobowe: rok (year), miesiąc (mo), dzień (da).
2. Lokalizacja stacji jest identyfikowana przez parę: stn + wban.
3. Kolumna country z tabeli stations opisuje kraj stacji.
4. Temp, max i min opisują temperatu

### krok 4

In [3]:
# 4.1 Lokalizacje i kraje
query_41 = """
SELECT DISTINCT s.usaf AS stn, s.wban, s.name, s.country, s.state
FROM `bigquery-public-data.noaa_gsod.stations` AS s
INNER JOIN (
    SELECT stn, wban FROM `bigquery-public-data.noaa_gsod.gsod1961`
    UNION DISTINCT
    SELECT stn, wban FROM `bigquery-public-data.noaa_gsod.gsod1962`
) AS active ON s.usaf = active.stn AND s.wban = active.wban
"""
df_41 = client.query(query_41).to_dataframe()
df_41 = df_41.replace(['99999', 99999], "brak").drop_duplicates()
df_41.to_csv('4_1_stacje.csv', index=False)

# 4.2 Warunki pogodowe dziennie
query_42 = """
SELECT stn, wban, year, mo, da, temp, dewp, slp, visib, wdsp, prcp
FROM `bigquery-public-data.noaa_gsod.gsod1961`
UNION ALL
SELECT stn, wban, year, mo, da, temp, dewp, slp, visib, wdsp, prcp
FROM `bigquery-public-data.noaa_gsod.gsod1962`
"""
df_42 = client.query(query_42).to_dataframe()
df_42['date'] = pd.to_datetime(
    df_42[['year', 'mo', 'da']].astype(str).agg('-'.join, axis=1)
)
df_42 = df_42.replace(['99999', 99999, 99.9, 99.99, 999.9, '999.9', 9999.9], "brak").drop_duplicates()
df_42.to_csv('4_2_pogoda_dzienna.csv', index=False)

# 4.3 Zjawiska ekstremalne
query_43 = """
SELECT stn, wban, year, mo, da, max, min, gust, mxpsd, sndp, hail, thunder, tornado_funnel_cloud
FROM `bigquery-public-data.noaa_gsod.gsod1961`
UNION ALL
SELECT stn, wban, year, mo, da, max, min, gust, mxpsd, sndp, hail, thunder, tornado_funnel_cloud
FROM `bigquery-public-data.noaa_gsod.gsod1962`
"""
df_43 = client.query(query_43).to_dataframe()
df_43['date'] = pd.to_datetime(
    df_43[['year', 'mo', 'da']].astype(str).agg('-'.join, axis=1)
)
df_43 = df_43.replace(['99999', 99999, 99.9, 99.99, 999.9, '999.9', 9999.9], "brak").drop_duplicates()
df_43.to_csv('4_3_ekstrema.csv', index=False)

# 4.4 Trendy czasowe
query_44 = """
SELECT stn, wban, year, mo, da, temp, prcp
FROM `bigquery-public-data.noaa_gsod.gsod1961`
UNION ALL
SELECT stn, wban, year, mo, da, temp, prcp
FROM `bigquery-public-data.noaa_gsod.gsod1962`
"""
df_44 = client.query(query_44).to_dataframe()
df_44['date'] = pd.to_datetime(
    df_44[['year', 'mo', 'da']].astype(str).agg('-'.join, axis=1)
)
df_44 = df_44.replace(['99999', 99999, 99.9, 99.99, 999.9, '999.9', 9999.9], "brak").drop_duplicates()
df_44.to_csv('4_4_trendy_czasowe.csv', index=False)

# 4.5 Analiza dni upalnych (temperatura maksymalna > 86 fahrenheit)
query_45 = """
SELECT stn, wban, year, mo, da, temp, max, min, visib
FROM `bigquery-public-data.noaa_gsod.gsod1961`
WHERE max > 86.0
UNION ALL
SELECT stn, wban, year, mo, da, temp, max, min, visib
FROM `bigquery-public-data.noaa_gsod.gsod1962`
WHERE max > 86.0
"""
df_45 = client.query(query_45).to_dataframe()
df_45['date'] = pd.to_datetime(
    df_45[['year', 'mo', 'da']].astype(str).agg('-'.join, axis=1)
)
df_45 = df_45.replace(['99999', 99999, 99.9, 99.99, 999.9, '999.9', 9999.9], "brak").drop_duplicates()
df_45.to_csv('4_5_dni_upalne.csv', index=False)

### krok 5

In [4]:
for df in [df_41, df_42, df_43, df_44, df_45]:
    df['stn'] = df['stn'].astype(str)
    df['wban'] = df['wban'].astype(str)

df_final = pd.merge(
    df_42, df_43, on=['stn', 'wban', 'date', 'year', 'mo', 'da'], how='outer')
df_final = pd.merge(df_final, df_44, on=[
                    'stn', 'wban', 'date', 'year', 'mo', 'da', 'temp', 'prcp'], how='outer')
df_final = pd.merge(df_final, df_45, on=[
                    'stn', 'wban', 'date', 'year', 'mo', 'da', 'temp', 'max', 'min', 'visib'], how='outer')
df_final = pd.merge(df_final, df_41, on=['stn', 'wban'], how='left')

df_final = df_final.drop_duplicates()

df_final.to_csv('5_finalny.csv', index=False)

### krok 6

In [5]:
df_fao = pd.read_csv('rolnictwo.csv', encoding='utf-8-sig')
df_fao.columns = df_fao.columns.str.strip()
df_final = pd.read_csv('5_finalny.csv', low_memory=False)

kolumny_pogodowe = ['temp', 'prcp', 'max', 'min', 'visib']
df_final[kolumny_pogodowe] = (
    df_final[kolumny_pogodowe]
    .replace('brak', np.nan)
    .apply(pd.to_numeric, errors='coerce')
)

df_final['year'] = pd.to_numeric(df_final['year'], errors='coerce').astype('Int64')
df_final = df_final.dropna(subset=['country', 'year'])

pogoda_roczna = df_final.groupby(['country', 'year'], as_index=False).agg({
    'temp': 'mean',
    'prcp': 'sum',
    'max': 'max',
    'min': 'min',
    'visib': 'mean'
})

mapa_krajow = {
    'AC': 'Antigua and Barbuda',
    'AE': 'United Arab Emirates',
    'AG': 'Algeria',
    'AJ': 'USSR',
    'AL': 'Albania',
    'AM': 'USSR',
    'AO': 'Angola',
    'AQ': 'Samoa',
    'AR': 'Argentina',
    'AS': 'Australia',
    'AU': 'Austria',
    'AY': 'Australia',
    'BA': 'Bahrain',
    'BD': 'United Kingdom of Great Britain and Northern Ireland',
    'BF': 'Bahamas',
    'BG': 'Bangladesh',
    'BH': 'Belize',
    'BK': 'Yugoslav SFR',
    'BL': 'Bolivia (Plurinational State of)',
    'BM': 'Myanmar',
    'BO': 'USSR',
    'BP': 'Solomon Islands',
    'BR': 'Brazil',
    'BU': 'Bulgaria',
    'CA': 'Canada',
    'CB': 'Cambodia',
    'CE': 'Sri Lanka',
    'CH': 'China, mainland',
    'CI': 'Chile',
    'CK': 'Australia',
    'CO': 'Colombia',
    'CQ': 'United States of America',
    'CU': 'Cuba',
    'CY': 'Cyprus',
    'DA': 'Denmark',
    'DJ': 'Djibouti',
    'EG': 'Egypt',
    'EN': 'USSR',
    'ER': 'Ethiopia PDR',
    'ES': 'El Salvador',
    'ET': 'Ethiopia PDR',
    'EZ': 'Czechoslovakia',
    'FI': 'Finland',
    'FJ': 'Fiji',
    'FM': 'Kiribati',
    'FR': 'France',
    'FS': 'France',
    'GG': 'USSR',
    'GI': 'United Kingdom of Great Britain and Northern Ireland',
    'GL': 'Denmark',
    'GM': 'Germany',
    'GP': 'Guadeloupe',
    'GQ': 'United States of America',
    'GR': 'Greece',
    'GT': 'Guatemala',
    'HO': 'Honduras',
    'HR': 'Yugoslav SFR',
    'HU': 'Hungary',
    'IC': 'Iceland',
    'ID': 'Indonesia',
    'IN': 'India',
    'IS': 'Israel',
    'IT': 'Italy',
    'JA': 'Japan',
    'JO': 'Jordan',
    'KE': 'Kenya',
    'KG': 'USSR',
    'KN': "Democratic People's Republic of Korea",
    'KR': 'Kiribati',
    'KS': 'Republic of Korea',
    'KU': 'Kuwait',
    'KZ': 'USSR',
    'LA': "Lao People's Democratic Republic",
    'LE': 'Lebanon',
    'LG': 'USSR',
    'LH': 'USSR',
    'LO': 'Czechoslovakia',
    'LQ': 'United States of America',
    'LY': 'Libya',
    'MB': 'Martinique',
    'MC': 'China, Macao SAR',
    'MD': 'USSR',
    'MG': 'Mongolia',
    'MJ': 'Yugoslav SFR',
    'MK': 'Yugoslav SFR',
    'ML': 'Mali',
    'MO': 'Morocco',
    'MQ': 'United States of America',
    'MR': 'Mauritania',
    'MT': 'Malta',
    'MU': 'Oman',
    'MY': 'Malaysia',
    'NG': 'Niger',
    'NL': 'Netherlands (Kingdom of the)',
    'NT': 'Netherlands (Kingdom of the)',
    'NU': 'Nicaragua',
    'NZ': 'New Zealand',
    'OD': 'Sudan (former)',
    'PE': 'Peru',
    'PK': 'Pakistan',
    'PL': 'Poland',
    'PM': 'Panama',
    'PN': 'United Kingdom of Great Britain and Northern Ireland',
    'PO': 'Portugal',
    'PS': 'Philippines',
    'RI': 'Yugoslav SFR',
    'RM': 'Kiribati',
    'RO': 'Romania',
    'RP': 'Philippines',
    'RQ': 'Puerto Rico',
    'RS': 'USSR',
    'SA': 'Saudi Arabia',
    'SE': 'Seychelles',
    'SF': 'South Africa',
    'SH': 'United Kingdom of Great Britain and Northern Ireland',
    'SI': 'Yugoslav SFR',
    'SN': 'Singapore',
    'SO': 'Somalia',
    'SP': 'Spain',
    'SU': 'Sudan (former)',
    'SV': 'Norway',
    'SY': 'Syrian Arab Republic',
    'SZ': 'Switzerland',
    'TH': 'Thailand',
    'TI': 'USSR',
    'TK': 'United Kingdom of Great Britain and Northern Ireland',
    'TS': 'Tunisia',
    'TT': 'Timor-Leste',
    'TW': 'China, Taiwan Province of',
    'TX': 'USSR',
    'TZ': 'United Republic of Tanzania',
    'UG': 'Uganda',
    'UK': 'United Kingdom of Great Britain and Northern Ireland',
    'UP': 'USSR',
    'US': 'United States of America',
    'UY': 'Uruguay',
    'UZ': 'USSR',
    'VE': 'Venezuela (Bolivarian Republic of)',
    'VM': 'Viet Nam',
    'VQ': 'United States of America',
    'WI': 'Morocco',
    'WQ': 'United States of America',
    'YM': 'Yemen',
    'ZI': 'Zimbabwe'
}

obszary_fao = set(df_fao['Area'].dropna().unique())
przefiltrowana = {}
for k, v in mapa_krajow.items():
    if v in obszary_fao:
        przefiltrowana[k] = v
mapa_krajow = przefiltrowana

pogoda_roczna['Area'] = pogoda_roczna['country'].map(mapa_krajow)
pogoda_roczna['Year'] = pd.to_numeric(pogoda_roczna['year'], errors='coerce').astype('Int64')
pogoda_roczna = pogoda_roczna.dropna(subset=['Area', 'Year'])

df_merged = pd.merge(
    df_fao,
    pogoda_roczna,
    on=['Area', 'Year'],
    how='inner'
).drop(columns=['country', 'year'], errors='ignore')

df_merged.to_csv('6_rolnictwo_pogoda.csv', index=False)